---
**Correlation & Clustering in Python**
Data Analysis Course · Week 3
---

This notebook is the Python equivalent of the R Markdown `_02_correlation_clustering.Rmd`.
Topics: **centrality measures**, **correlation** (Pearson vs. Spearman), and **k-means clustering**
(with WSS / elbow method and the silhouette method).

Work through it cell by cell — run each code cell with **Shift+Enter**.

**Required packages:** `pandas`, `numpy`, `matplotlib`, `scipy`, `scikit-learn`, `pyreadr`
```
pip install pandas numpy matplotlib scipy scikit-learn pyreadr
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
from scipy.spatial.distance import pdist, squareform

We start again from the cleaned diabetes dataset (same steps as Week 2).

In [ ]:
dat = pd.read_csv("https://tinyurl.com/y4fark9g", sep="\t")
dat = dat.set_index("id")

cols_to_remove = ["bp.2s", "bp.2d", "time.ppn"]
dat = dat.drop(columns=cols_to_remove)

column_order = [
    "gender", "location", "frame", "chol",
    "stab.glu", "hdl", "ratio", "glyhb", "age",
    "height", "weight", "bp.1s", "bp.1d"
]
dat = dat[column_order]

nb_na_rows = dat.isna().sum(axis=1)
i_missing = nb_na_rows[nb_na_rows > 0].index
dat = dat.drop(index=i_missing)
dat.head()

## 1 – Measuring centrality *(equivalent to R section 2)*

### Mean

In [ ]:
dat["stab.glu"].mean()   # R: mean(dat$stab.glu)

### Median

In [ ]:
dat["stab.glu"].median()   # R: median(dat$stab.glu)

# Calculate the mean and median of other continuous variables and compare them.
# (a) Why is there a difference between mean and median?
# (b) Why do you think it's larger for some variables and almost zero for others?

### Quantiles

In [ ]:
dat["stab.glu"].quantile([0, 0.25, 0.5, 0.75, 1])   # R: quantile(dat$stab.glu)

# Did you remember how it works? How can you pick any specific quantile you need?

## 2 – Association between variables *(equivalent to R section 3)*

A common step during any data analysis project is to find associations between variables. In our
diabetes dataset we would expect, for instance, a high correlation between blood glucose levels and
glycosylated hemoglobin, or between waist and hip size.

### Correlations

In [ ]:
# Scatter plot between two variables
plt.scatter(dat["stab.glu"], dat["glyhb"], s=10)
plt.xlabel("Stabilized glucose")
plt.ylabel("Glycosylated hemoglobin")
plt.show()

# Do you suspect a relationship between the two variables?
# Try the scatter plot for other pairs of numerical variables!

In [ ]:
# R: cor(dat$stab.glu, dat$glyhb, method='pearson')
pearson_r = dat["stab.glu"].corr(dat["glyhb"], method="pearson")
print("Pearson: ", pearson_r)

# R: cor(dat$stab.glu, dat$glyhb, method='spearman')
spearman_r = dat["stab.glu"].corr(dat["glyhb"], method="spearman")
print("Spearman:", spearman_r)

# Comment on the difference between the two measures!
# Redo the analysis for the variables hip and waist (note: not present in this dataset version).

### Pairwise scatter plot

In [ ]:
# The numerical columns are chol, stab.glu, hdl, ratio, glyhb, age, height, weight, bp.1s, bp.1d
# Can you build a DataFrame dat_num containing only the numerical columns?
numeric_cols = ["chol", "stab.glu", "hdl", "ratio", "glyhb", "age", "height", "weight", "bp.1s", "bp.1d"]
dat_num = dat[numeric_cols]

In [ ]:
# R: pairs(dat.num, pch=20, cex=0.5, col="grey")
pd.plotting.scatter_matrix(dat_num, figsize=(12, 10), s=5, color="grey")
plt.show()

# Which correlations did you expect, and which were novel?
# Can you explain the relation between hdl, chol and ratio?
# Remember: ratio = chol / hdl

> Associations are among the simplest forms of structure in data. Remember: *association does not
> imply correlation*, and *correlation does not imply causation*.

## 3 – Unsupervised learning: clustering *(equivalent to R section 4)*

**Unsupervised clustering** identifies groups (clusters) of observations or variables without using
any prior knowledge about group membership. **K-means clustering** is a good example: samples are
grouped based uniquely on the data itself.

We now switch to a gene expression dataset from the TCGA (The Cancer Genome Atlas) project: 200
samples (patients, columns) × 300 genes (rows).

### 3.1 – Load data

In [ ]:
import pyreadr, urllib.request

# R's readRDS() has no direct pandas equivalent — we download the .rds file and read it with pyreadr
urllib.request.urlretrieve(
    "https://www.dropbox.com/scl/fi/xlkfv97po3yq2qg9ekftd/brca.rds?rlkey=3c8d0lktsrfs4989rpaenm1qq&dl=1",
    "brca.rds"
)
brca_exp = pyreadr.read_r("brca.rds")[None]   # genes x samples, like in R
print(brca_exp.shape)          # R: dim(brca.exp)
brca_exp.iloc[:10, :10]        # R: brca.exp[1:10,1:10]

**If you have trouble loading the data**, download [this file](https://www.dropbox.com/s/qububmfvtv443mq/brca.exp.rds?dl=1),
store it locally, and load it with:
```python
# brca_exp = pyreadr.read_r("brca.exp.rds")[None]
```

In [ ]:
urllib.request.urlretrieve(
    "https://www.dropbox.com/s/9xlivejqkj77llc/brca.anno.rds?dl=1",
    "brca_anno.rds"
)
brca_anno = pyreadr.read_r("brca_anno.rds")[None]
brca_anno.head()

In [ ]:
# R: table(brca.anno$HER2_status)
brca_anno["HER2_status"].value_counts()

### 3.2 – k-means clustering

In [ ]:
# R: km = kmeans(x=t(brca.exp), centers=2, nstart=10)
# scikit-learn clusters ROWS, so we transpose to cluster SAMPLES (columns) like in R
X = brca_exp.T.values
km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)

print(km.labels_)                          # cluster assignment per sample
print(pd.Series(km.labels_).value_counts())   # R: table(km$cluster)

# Play around with n_clusters. What do you observe?

### Quality of the clustering

In [ ]:
# R: km$tot.withinss
print(km.inertia_)   # scikit-learn calls WSS "inertia_"

# What would WSS be if the number of clusters equals the number of data points?

In [ ]:
# R: wss = sapply(2:7, function(k) kmeans(...)$tot.withinss)
wss = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X).inertia_ for k in range(2, 8)]

plt.plot(range(2, 8), wss, marker="o")
plt.xlabel("Number of clusters K")
plt.ylabel("Total within-clusters sum of squares")
plt.show()

# Do you see an obvious "elbow" or "kink" in the curve?

In [ ]:
# R: D = dist(t(brca.exp))
D = squareform(pdist(X))   # pairwise sample-sample distance matrix

In [ ]:
# R: km = kmeans(..., centers=3); s = silhouette(km$cluster, D); plot(s)
km3 = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
sil_values = silhouette_samples(X, km3.labels_)
sil_avg = silhouette_score(X, km3.labels_)
print("Average silhouette:", sil_avg)

# Sort silhouette values within each cluster for a silhouette plot
fig, ax = plt.subplots()
y_lower = 0
for cluster in sorted(set(km3.labels_)):
    vals = np.sort(sil_values[km3.labels_ == cluster])
    y_upper = y_lower + len(vals)
    ax.barh(range(y_lower, y_upper), vals, height=1.0)
    y_lower = y_upper
ax.axvline(sil_avg, color="red", linestyle="--")
ax.set_xlabel("Silhouette coefficient")
ax.set_ylabel("Samples (grouped by cluster)")
plt.show()

# Check the average silhouette value; repeat for other values of k.

---
## Exercises

### Exercise 1

1. Compute the mean cholesterol value and store it in `m`.
2. Compute the 1%, 10%, 25%, 50% (median), 75%, 90%, 99% quantiles using `.quantile([0.01, 0.1, ...])`.
   Store the result in `q`.
3. Plot cholesterol as a histogram using `plt.hist()`.
4. Use `plt.axvline(x=...)` to draw vertical lines at each quantile on top of the histogram.
5. Repeat this with blood pressure!

In [ ]:
# Your code here:

### Exercise 2

1. Scatter-plot `hip` and `waist` and compute their correlation (note: use `dat_num` columns available in your dataset).
2. Apply `.corr()` **directly** on `dat_num`: `dat_num.corr()`. What is the output? Store it in `all_cor`.
3. Find the highest and lowest Pearson correlation values (excluding the diagonal). Which variable
   pairs do they correspond to? Plot the corresponding scatter plots!

In [ ]:
# Your code here:

### Exercise 3

Apply the skills from the previous exercise sheet to the gene expression data.

1. Calculate the min, max and median expression level for each **sample** (column). *Hint: `brca_exp.min(axis=0)`, etc.*
2. Repeat the same for each **gene** (row).
3. Plot a histogram/density plot of the median expression values for samples and for genes.
4. Calculate how many genes have a lower standard deviation of expression than the average standard
   deviation across all genes. Print the count.

In [ ]:
# Your code here:

### Going further...

A larger version of this dataset, with more genes for the same samples, is available:
```python
urllib.request.urlretrieve("https://www.dropbox.com/s/m6nfxul95borupz/brca.exp_large.rds?dl=1", "brca_large.rds")
brca_large = pyreadr.read_r("brca_large.rds")[None]
```

1. Compute the standard deviation of each gene (row) using `.std(axis=1)`.
2. Use `.sort_values(ascending=False)` to find the row index of the top 200 most variant genes.
3. Redo the analysis with these genes.

In [ ]:
# Your code here:

## Summary: What have we learned?

| R | Python | Purpose |
|---|--------|---------|
| `mean(x)` / `median(x)` | `x.mean()` / `x.median()` | Central tendency |
| `quantile(x)` | `x.quantile([...])` | Quantiles |
| `cor(x, y, method='pearson')` | `x.corr(y, method='pearson')` | Pearson correlation |
| `cor(x, y, method='spearman')` | `x.corr(y, method='spearman')` | Spearman correlation |
| `pairs(df)` | `pd.plotting.scatter_matrix(df)` | Pairwise scatter plots |
| `cor(df)` | `df.corr()` | Correlation matrix |
| `readRDS(url(...))` | `pyreadr.read_r("file.rds")` (after download) | Load an .rds file |
| `kmeans(x, centers=k)` | `KMeans(n_clusters=k).fit(X)` | K-means clustering |
| `km$tot.withinss` | `km.inertia_` | Within-cluster sum of squares (WSS) |
| `dist(x)` | `scipy.spatial.distance.pdist/squareform` | Pairwise distance matrix |
| `silhouette(cluster, D)` | `sklearn.metrics.silhouette_samples/score` | Silhouette method |